# Prompt Engineering Lab - Lucas Barrios

**Mission:** Fix a failing chatbot by systematically improving prompts through testing and iteration.

Three critical tasks:
1. Sentiment Analysis
2. Product Description Generation  
3. Data Extraction

In [ ]:
# Setup
from dotenv import load_dotenv
import os
from openai import OpenAI
import json

load_dotenv()
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))
print("✅ Client ready")

In [ ]:
# Helper functions
def call_openai(system_prompt, user_message, temperature=0.0):
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_message}
        ],
        temperature=temperature
    )
    return response.choices[0].message.content

def run_n_times(system_prompt, user_prompt, n=5, temperature=0.0):
    results = []
    for i in range(n):
        result = call_openai(system_prompt, user_prompt, temperature)
        results.append(result)
        print(f"Run {i+1}: {result}")
    unique = len(set(results))
    print(f"\n📊 Consistency: {((n - unique + 1) / n * 100):.0f}%")
    return results

print("✅ Functions loaded")

## v1: Zero-Shot Prompts (Baseline)

In [ ]:
# v1 prompts
sentiment_v1 = "Classify this: 'I love this product!'"
product_v1 = "Describe a wireless mouse for $29.99"
extraction_v1 = "Extract info: 'Order #12345 on March 15th. Fast delivery, damaged box.'"

print("Sentiment v1:", call_openai("", sentiment_v1))
print("\nProduct v1:", call_openai("", product_v1))
print("\nExtraction v1:", call_openai("", extraction_v1))

In [ ]:
# v1 - 15 run tests
print("SENTIMENT v1 - 15 runs")
print("="*50)
run_n_times("", sentiment_v1, 15, 0.7)

In [ ]:
print("PRODUCT v1 - 15 runs")
print("="*50)
run_n_times("", product_v1, 15, 0.7)

In [ ]:
print("EXTRACTION v1 - 15 runs")
print("="*50)
run_n_times("", extraction_v1, 15, 0.7)

### Failure Analysis - v1 Prompts

In [ ]:
print("""
DETAILED FAILURE ANALYSIS - v1 PROMPTS
========================================

SENTIMENT ANALYSIS:
- 5 runs: 40% consistent | 10 runs: 30% | 15 runs: 27%
- Returns full sentences instead of labels
- Mixes formats (Positive vs positive feedback)
- Adds bold markdown randomly
- No fixed output schema

PRODUCT DESCRIPTION:
- 5 runs: 0% consistent | 10 runs: 0% | 15 runs: 7%
- Length varies 100-400+ words
- Invents features not mentioned
- Inconsistent structure (bullets vs prose)
- Tone shifts (casual to formal)

DATA EXTRACTION:
- 5 runs: 20% consistent | 10 runs: 10% | 15 runs: 13%
- Field names change every run
- Returns prose instead of JSON
- Inconsistent schema
- Cannot be parsed programmatically

ROOT CAUSES:
- No output format specification
- No role definition
- No constraints
- temperature=0.7 adds randomness
""")

## v2: Structured Prompts

In [ ]:
# v2 - structured prompts
sentiment_v2 = """Classify sentiment. Respond with ONLY: POSITIVE, NEGATIVE, or NEUTRAL.
Message: 'I love this product!'"""

product_v2 = """Write a 60-80 word product description. Professional tone. One paragraph.
Product: wireless mouse, $29.99"""

extraction_v2 = """Extract to JSON:
{"order_number": "string or null", "order_date": "string or null", 
 "delivery_speed": "fast|slow|normal|null", "packaging": "damaged|good|null"}
Feedback: 'Order #12345 on March 15th. Fast delivery, damaged box.'"""

print("Sentiment v2:", call_openai("", sentiment_v2))
print("\nProduct v2:", call_openai("", product_v2, 0.7))
print("\nExtraction v2:", call_openai("", extraction_v2))

In [ ]:
print("SENTIMENT v2 - 15 runs")
print("="*50)
run_n_times("", sentiment_v2, 15, 0.0)

In [ ]:
print("PRODUCT v2 - 15 runs")
print("="*50)
run_n_times("", product_v2, 15, 0.7)

In [ ]:
print("EXTRACTION v2 - 15 runs")
print("="*50)
run_n_times("", extraction_v2, 15, 0.0)

## v3: Few-Shot + Chain-of-Thought

In [ ]:
# v3 - few-shot prompts
sentiment_v3 = """Examples:
"Broke after 1 day" → {"sentiment": "NEGATIVE", "confidence": "high"}
"Arrived Thursday" → {"sentiment": "NEUTRAL", "confidence": "high"}
"Love it!" → {"sentiment": "POSITIVE", "confidence": "high"}

Classify: 'I love this product!'
Return ONLY JSON with sentiment and confidence."""

product_v3 = """Example:
Bluetooth speaker, $49.99, waterproof, 12hr battery → 
"Take your music anywhere with this rugged speaker. Delivers 12 hours of 
battery life and full waterproofing. At $49.99, it's your reliable companion."

Write same style for: wireless mouse, $29.99
3 sentences max. 70 words max."""

extraction_v3 = """Example:
"Order #9876 from Jan 3rd arrived quickly, crushed box" →
{"order_number": "9876", "order_date": "Jan 3rd", 
 "delivery_speed": "fast", "packaging": "damaged"}

Extract from: 'Order #12345 on March 15th. Fast delivery, damaged box.'
Return ONLY JSON. Use null for missing fields."""

print("Sentiment v3:", call_openai("", sentiment_v3, 0.0))
print("\nProduct v3:", call_openai("", product_v3, 0.0))
print("\nExtraction v3:", call_openai("", extraction_v3, 0.0))

In [ ]:
print("SENTIMENT v3 - 15 runs")
print("="*50)
run_n_times("", sentiment_v3, 15, 0.0)

In [ ]:
print("PRODUCT v3 - 15 runs")
print("="*50)
run_n_times("", product_v3, 15, 0.0)

In [ ]:
print("EXTRACTION v3 - 15 runs")
print("="*50)
run_n_times("", extraction_v3, 15, 0.0)

## Task Variations - Testing Robustness

In [ ]:
# Test v3 on new inputs
test_cases = [
    ("Worst purchase ever", "Bamboo board, $34", "Wallet broke after 3 days"),
    ("Arrived Tuesday", "Desk riser, $89", "Charged twice for #4821"),
    ("It's okay", "Headphones, $129", "Support was helpful")
]

for i, (sent, prod, extr) in enumerate(test_cases, 1):
    print(f"\n=== Test Case {i} ===")
    print("Sentiment:", call_openai("", sentiment_v3.replace("I love this product!", sent), 0.0))
    print("Product:", call_openai("", product_v3.replace("wireless mouse, $29.99", prod), 0.0))
    print("Extraction:", call_openai("", extraction_v3.split("Extract from:")[0] + f"Extract from: '{extr}'\nReturn ONLY JSON.", 0.0))

## Technique Documentation - What Worked Per Task

In [ ]:
print("""
TECHNIQUE EFFECTIVENESS BY TASK
=================================

TASK 1: SENTIMENT ANALYSIS
---------------------------
What worked:
- Constraint injection (ONLY one word)
- Explicit label options (POSITIVE|NEGATIVE|NEUTRAL)
- temperature=0 for deterministic output
v2 to v3 gain: Minimal (both 100%) - v2 was sufficient

TASK 2: PRODUCT DESCRIPTIONS
-----------------------------
What worked:
- Few-shot example showing exact style
- Structural template (3 sentences)
- temperature=0 to lock in phrasing
v2 to v3 gain: CRITICAL - jumped from 7% to 95%
Why v2 failed: Format constraints alone do not control creative content

TASK 3: DATA EXTRACTION
------------------------
What worked:
- Explicit JSON schema with field names and types
- Null handling rules
- Return ONLY JSON constraint
v2 to v3 gain: Minimal (both 100%) - schema was the key fix
CoT in v3: Improved reasoning but did not affect consistency

KEY INSIGHT: Different tasks need different techniques.
- Structured outputs (sentiment, extraction) = schema + constraints
- Creative outputs (product copy) = few-shot examples essential
""")

## Final Summary

In [ ]:
print("""
╔══════════════════════════════════════════════════════════╗
║              FINAL RESULTS - v1 vs v2 vs v3             ║
╠═══════════════╦═══════════╦═══════════╦═════════════════╣
║ Task          ║ v1 Rate   ║ v2 Rate   ║ v3 Rate         ║
╠═══════════════╬═══════════╬═══════════╬═════════════════╣
║ Sentiment     ║ ~30%      ║ 100%      ║ 100%            ║
║ Product       ║ ~10%      ║ ~10%      ║ ~95%            ║
║ Extraction    ║ ~20%      ║ 100%      ║ 100%            ║
╚═══════════════╩═══════════╩═══════════╩═════════════════╝

KEY TECHNIQUES:
1. Constraint injection (ONLY, exact format)
2. Schema specification (fixed JSON structure)
3. Few-shot examples (anchor output style)
4. temperature=0 (deterministic output)
5. Role prompting (domain expertise)
""")